# Vetch 0.1.7 PUE Testing - Anthropic Claude

This notebook tests the provider-specific PUE implementation with Anthropic Claude.

**Expected behavior:**
- Claude models should detect `pue=1.15` (AWS-backed)
- Event should include `pue_tier=1` and `pue_source="AWS Sustainability Report 2024 (AWS-backed)"`

In [ ]:
# Install Vetch (if testing from local dev)
# !pip install -e .

import os
from pprint import pprint

In [ ]:
# Set your Anthropic API key
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

# Verify key is set
assert "ANTHROPIC_API_KEY" in os.environ, "Please set ANTHROPIC_API_KEY environment variable"

## Test 1: Basic PUE Detection with Claude

In [ ]:
import vetch
from anthropic import Anthropic

client = Anthropic()

# Use Vetch wrapper with region
with vetch.wrap(region="us-east-1", tags={"test": "pue-detection"}) as ctx:
    response = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=100,
        messages=[{"role": "user", "content": "Hello, Claude! Please respond in exactly 20 words."}]
    )
    
    print(f"Response: {response.content[0].text}")

# Check event data
event = ctx.event
print(f"\nModel: {event['model']}")
print(f"Provider: {event['provider']}")
print(f"Energy: {event['estimated_energy_wh']} Wh")
print(f"Carbon: {event['estimated_carbon_g']} gCO2e")
print(f"\n--- PUE Metadata (NEW in 0.1.7) ---")
print(f"PUE: {event['pue']} (expected: 1.15 for AWS-backed Anthropic)")
print(f"PUE Tier: {event['pue_tier']} (1=known value)")
print(f"PUE Source: {event['pue_source']}")

# Assertions
assert event['pue'] == 1.15, f"Expected PUE=1.15 for Claude, got {event['pue']}"
assert event['pue_tier'] == 1, f"Expected PUE tier=1 (known), got {event['pue_tier']}"
assert 'AWS' in event['pue_source'], f"Expected AWS in source, got {event['pue_source']}"

print("\n✅ All assertions passed!")

## Test 2: Global Instrumentation

In [ ]:
# Reset client for clean test
import vetch
from anthropic import Anthropic

# Use global instrumentation
vetch.instrument(region="eu-west-1", tags={"service": "test"})

client = Anthropic()

# Make a call - automatically tracked
with vetch.wrap() as ctx:
    response = client.messages.create(
        model="claude-3-haiku-20240307",
        max_tokens=50,
        messages=[{"role": "user", "content": "What is 2+2?"}]
    )
    
    print(f"Response: {response.content[0].text}")

# Verify event
event = ctx.event
print(f"\nModel: {event['model']}")
print(f"PUE: {event['pue']} (Claude Haiku should also be 1.15)")
print(f"Region: {event['region']}")

assert event['pue'] == 1.15
print("\n✅ Global instrumentation working!")

# Clean up
vetch.uninstrument()

## Test 3: User Override PUE

In [ ]:
import os
import vetch
from anthropic import Anthropic

# Override PUE with custom value
os.environ["VETCH_DEFAULT_PUE"] = "1.05"

client = Anthropic()

with vetch.wrap(region="us-west-2") as ctx:
    response = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=50,
        messages=[{"role": "user", "content": "Hi!"}]
    )

event = ctx.event
print(f"PUE: {event['pue']} (expected: 1.05 from user config)")
print(f"PUE Tier: {event['pue_tier']} (expected: 1 for user config)")
print(f"PUE Source: {event['pue_source']}")

assert event['pue'] == 1.05, f"User override failed, got {event['pue']}"
assert event['pue_tier'] == 1, f"User config should be Tier 1, got {event['pue_tier']}"
assert 'user config' in event['pue_source'].lower()

print("\n✅ User PUE override working!")

# Clean up
del os.environ["VETCH_DEFAULT_PUE"]

## Test 4: Full Event Inspection

In [ ]:
import vetch
from anthropic import Anthropic
import json

client = Anthropic()

with vetch.wrap(region="us-east-1", tags={"test": "full-event"}) as ctx:
    response = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=100,
        messages=[{"role": "user", "content": "Explain quantum computing in one sentence."}]
    )

# Pretty-print full event
print("Full Event JSON:")
print(json.dumps(ctx.event, indent=2, default=str))

# Highlight PUE fields
print("\n--- PUE Fields ---")
print(f"pue: {ctx.event['pue']}")
print(f"pue_tier: {ctx.event['pue_tier']}")
print(f"pue_source: {ctx.event['pue_source']}")

## Test 5: Streaming Support

In [ ]:
import vetch
from anthropic import Anthropic

client = Anthropic()

with vetch.wrap(region="us-east-1") as ctx:
    stream = client.messages.create(
        model="claude-3-haiku-20240307",
        max_tokens=100,
        messages=[{"role": "user", "content": "Count from 1 to 10."}],
        stream=True
    )
    
    print("Streaming response:")
    for chunk in stream:
        if hasattr(chunk, 'delta') and hasattr(chunk.delta, 'text'):
            print(chunk.delta.text, end='', flush=True)
    print()

# Verify PUE is tracked even for streaming
event = ctx.event
print(f"\nStream completed")
print(f"PUE: {event['pue']} (should be 1.15 even for streaming)")
print(f"Energy: {event['estimated_energy_wh']} Wh")
print(f"Is Stream: {event['is_stream']}")

assert event['is_stream'] == True
assert event['pue'] == 1.15

print("\n✅ Streaming with PUE tracking working!")

## Summary

This notebook tested:
1. ✅ Automatic PUE detection for Claude models (1.15 for AWS-backed)
2. ✅ Global instrumentation with PUE tracking
3. ✅ User PUE override via `VETCH_DEFAULT_PUE`
4. ✅ Full event schema with new PUE fields
5. ✅ Streaming support with PUE metadata

**New in 0.1.7:**
- `event['pue']`: Datacenter efficiency value (1.10-1.15 for major clouds)
- `event['pue_tier']`: Confidence tier (1=known, 3=default)
- `event['pue_source']`: Data provenance (vendor report or user config)

All tests should show PUE=1.15 for Anthropic Claude (AWS-backed) unless overridden by user config.

In [ ]:
# Note: This test uses local calculation, not live API
# We'll use the calculation module directly to show the non-linear behavior

from vetch.calculation import calculate_energy

print("=== Non-Linear Energy Model Demo ===\n")
print("Testing GPT-4o with different prompt lengths")
print("(Based on Jegham et al. 2025 hardware measurements)\n")

# Short prompt (< 1000 tokens)
short_energy, tier, uncertainty, source, basis, known = calculate_energy(
    input_tokens=100,
    output_tokens=300,
    model="gpt-4o"
)

# Medium prompt (1000-5000 tokens)
medium_energy, _, _, _, _, _ = calculate_energy(
    input_tokens=1000,
    output_tokens=1000,
    model="gpt-4o"
)

# Long prompt (> 5000 tokens)
long_energy, _, _, _, _, _ = calculate_energy(
    input_tokens=10000,
    output_tokens=1500,
    model="gpt-4o"
)

print(f"Short prompt (400 tokens total):")
print(f"  Energy: {short_energy:.4f} Wh")
print(f"  Per 1k tokens: {(short_energy / 0.4):.4f} Wh/1k")
print(f"  Tier: {tier} (measured data)")
print()

print(f"Medium prompt (2000 tokens total):")
print(f"  Energy: {medium_energy:.4f} Wh")
print(f"  Per 1k tokens: {(medium_energy / 2.0):.4f} Wh/1k")
print()

print(f"Long prompt (11500 tokens total):")
print(f"  Energy: {long_energy:.4f} Wh")
print(f"  Per 1k tokens: {(long_energy / 11.5):.4f} Wh/1k")
print()

# Calculate efficiency gain
short_per_1k = short_energy / 0.4
long_per_1k = long_energy / 11.5
efficiency_gain = short_per_1k / long_per_1k

print(f"✨ Key Finding: Long prompts are {efficiency_gain:.1f}x more efficient per token!")
print(f"   (Due to amortization of fixed overhead costs)")
print()
print(f"Source: {basis}")
print()
print("This is why batch processing and long-context models are more energy-efficient.")

## Test 6: Non-Linear Energy Model (NEW in 0.1.7!)

Vetch 0.1.7 introduces **prompt-length-aware energy coefficients** based on Jegham et al. (2025) measurements.
Energy consumption is NOT linear with token count - longer prompts are more efficient per token due to amortization of fixed costs.

This test demonstrates how the same model can have different energy efficiency depending on prompt length.